# sqlite3

对应 `stdlib.md`：内置 SQLite。

笔记本在 `python_base/sqlite3/qa.ipynb`。数据库文件落在题目子目录里，路径相对 `ROOT`。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

说明：SQL 占位符用 `?`，不要把值拼进语句。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/sqlite3。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "sqlite3" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "sqlite3"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/sqlite3')

## 1. 建表并插入一行

在 `db` 里建表 `people(name TEXT, score INTEGER)`，插入 `("Ada", 90)`。查出这一行的 `name` 并打印。


In [ ]:
box = ROOT / "q1"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)

# 作答

import sqlite3

with sqlite3.connect(db) as conn:
    # CREATE TABLE：建表。TEXT / INTEGER 是列类型。
    conn.execute("CREATE TABLE people(name TEXT, score INTEGER)")
    # INSERT：插入一行。? 是占位符，值放在后面的元组里。
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Ada", 90))
    # SELECT：查询。fetchone() 取一行，结果是 (name, score)。
    row = conn.execute("SELECT name FROM people").fetchone()
    print(row[0])


## 2. 用占位符查询

表 `people` 已经有三行。用占位符查出 `score` 大于等于 `80` 的 `name`，按查出的顺序打印。


In [ ]:
import sqlite3

box = ROOT / "q2"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)
with sqlite3.connect(db) as conn:
    conn.execute("CREATE TABLE people(name TEXT, score INTEGER)")
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Ada", 90))
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Bob", 70))
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Cy", 88))

# 作答

with sqlite3.connect(db) as conn:
    # WHERE score >= ? ：筛选。? 换成 80，不要写成 f"...{80}"。
    rows = conn.execute(
        "SELECT name FROM people WHERE score >= ?",
        (80,),
    ).fetchall()
    for row in rows:
        print(row[0])


## 3. 一次插入多行

表 `people(name TEXT, score INTEGER)` 是空的。把 `rows` 一次插入。然后打印表里的行数。


In [ ]:
import sqlite3

box = ROOT / "q3"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)
rows = [("Ada", 90), ("Bob", 70)]
with sqlite3.connect(db) as conn:
    conn.execute("CREATE TABLE people(name TEXT, score INTEGER)")

# 作答

with sqlite3.connect(db) as conn:
    # executemany：同一条 INSERT 喂多行。
    conn.executemany("INSERT INTO people(name, score) VALUES (?, ?)", rows)
    # COUNT(*)：行数。
    print(conn.execute("SELECT COUNT(*) FROM people").fetchone()[0])


## 4. 提交后重开还能读到

向空表 `people(name TEXT)` 插入 `"Ada"` 并提交。关掉连接后重新打开同一个 `db`，查出 `name` 并打印。


In [ ]:
box = ROOT / "q4"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)

# 作答

import sqlite3

conn = sqlite3.connect(db)
conn.execute("CREATE TABLE people(name TEXT)")
conn.execute("INSERT INTO people(name) VALUES (?)", ("Ada",))
conn.commit()  # 不 commit，关掉后数据不会留下
conn.close()

conn = sqlite3.connect(db)
print(conn.execute("SELECT name FROM people").fetchone()[0])
conn.close()


## 5. 按列名取值

表里已有 `("Ada", 90)`。查出这一行后，用列名 `name` 取值并打印（不要用下标 `0`）。


In [ ]:
import sqlite3

box = ROOT / "q5"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)
with sqlite3.connect(db) as conn:
    conn.execute("CREATE TABLE people(name TEXT, score INTEGER)")
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Ada", 90))

# 作答

with sqlite3.connect(db) as conn:
    # Row 让结果能用列名取值：row["name"]，而不只是 row[0]。
    conn.row_factory = sqlite3.Row
    row = conn.execute("SELECT name, score FROM people").fetchone()
    print(row["name"])


## 6. 更新一行

表里已有 `("Ada", 90)`。把 `Ada` 的 `score` 改成 `100`。再查出她的 `score` 并打印。


In [ ]:
import sqlite3

box = ROOT / "q6"
box.mkdir(parents=True, exist_ok=True)
db = box / "lab.db"
db.unlink(missing_ok=True)
with sqlite3.connect(db) as conn:
    conn.execute("CREATE TABLE people(name TEXT, score INTEGER)")
    conn.execute("INSERT INTO people(name, score) VALUES (?, ?)", ("Ada", 90))

# 作答

with sqlite3.connect(db) as conn:
    # UPDATE ... SET ... WHERE：改符合条件的行。
    conn.execute("UPDATE people SET score = ? WHERE name = ?", (100, "Ada"))
    print(conn.execute("SELECT score FROM people WHERE name = ?", ("Ada",)).fetchone()[0])
